In [1]:
import pandas as pd

from src.data_loader import load_data
import matplotlib.pyplot as plt
from pathlib import Path
import plotly.express as px

In [2]:
proj_path = Path.cwd().parent
data_path = proj_path / 'data'

In [3]:
df = load_data(data_path/'train.csv')
df.head(10)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680
5,6,CA-2015-115812,09/06/2015,14/06/2015,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032.0,West,FUR-FU-10001487,Furniture,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.8600
6,7,CA-2015-115812,09/06/2015,14/06/2015,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032.0,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.2800
7,8,CA-2015-115812,09/06/2015,14/06/2015,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032.0,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.1520
8,9,CA-2015-115812,09/06/2015,14/06/2015,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032.0,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by S...,18.5040
9,10,CA-2015-115812,09/06/2015,14/06/2015,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032.0,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9000


In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 27 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Row ID            9800 non-null   int64         
 1   Order ID          9800 non-null   str           
 2   Order Date        9800 non-null   datetime64[us]
 3   Ship Date         9800 non-null   datetime64[us]
 4   Ship Mode         9800 non-null   str           
 5   Customer ID       9800 non-null   str           
 6   Customer Name     9800 non-null   str           
 7   Segment           9800 non-null   str           
 8   Country           9800 non-null   str           
 9   City              9800 non-null   str           
 10  State             9800 non-null   str           
 11  Postal Code       9789 non-null   str           
 12  Region            9800 non-null   str           
 13  Product ID        9800 non-null   str           
 14  Category          9800 non-null   s

In [ ]:
df.columns

Converting date columns to datetime type and postal code to str

In [5]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d/%m/%Y')
df['Postal Code'] = df['Postal Code'].astype(str).replace('.0', '', regex=False)

Adding shipping time column

In [6]:
df['shipping_time'] = (df['Ship Date'] - df['Order Date']).dt.days

Adding year, month, weekday and day columns (for order and ship date)

In [28]:
df['Order_month'], df['Order_day'], df['Order_year'], df['Order_weekday'] = df['Order Date'].dt.month, df['Order Date'].dt.day, df['Order Date'].dt.year, df['Order Date'].dt.weekday
df['Ship_month'], df['Ship_day'], df['Ship_year'] = df['Ship Date'].dt.month, df['Ship Date'].dt.day, df['Ship Date'].dt.year

df['Order year_month'] = df['Order_year'].astype(str) + '-' + df['Order_month'].astype(str)
df['Order year_month'] = pd.to_datetime(df['Order year_month'], format='%Y-%m')

In [9]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sales,shipping_time,Order_month,Order_day,Order_year,Order_weekday,Ship_month,Ship_day,Ship_year,Order year_month
0,1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,261.9600,3,11,8,2017,2,11,11,2017,2017-11
1,2,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,731.9400,3,11,8,2017,2,11,11,2017,2017-11
2,3,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,14.6200,4,6,12,2017,0,6,16,2017,2017-6
3,4,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,957.5775,7,10,11,2016,1,10,18,2016,2016-10
4,5,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,22.3680,7,10,11,2016,1,10,18,2016,2016-10


In [36]:
df.describe()

,Row ID,Order Date,Ship Date,Sales,shipping_time,Order_month,Order_day,Order_year,Order_weekday,Ship_month,Ship_day,Ship_year,Order year_month
count,9800.000000,9800,9800,9800.000000,9800.000000,9800.000000,9800.000000,9800.000000,9800.000000,9800.000000,9800.000000,9800.000000,9800
mean,4900.500000,2017-05-01 05:13:51.673469,2017-05-05 04:17:52.653061,230.769059,3.961122,7.818469,15.486837,2016.724184,2.993673,7.752245,15.895816,2016.739388,2017-04-16 17:32:48.979591
min,1.000000,2015-01-03 00:00:00,2015-01-07 00:00:00,0.444000,0.000000,1.000000,1.000000,2015.000000,0.000000,1.000000,1.000000,2015.000000,2015-01-01 00:00:00
25%,2450.750000,2016-05-24 00:00:00,2016-05-27 18:00:00,17.248000,3.000000,5.000000,8.000000,2016.000000,1.000000,5.000000,8.000000,2016.000000,2016-05-01 00:00:00
50%,4900.500000,2017-06-26 00:00:00,2017-06-29 00:00:00,54.490000,4.000000,9.000000,16.000000,2017.000000,3.000000,9.000000,16.000000,2017.000000,2017-06-01 00:00:00
75%,7350.250000,2018-05-15 00:00:00,2018-05-19 00:00:00,210.605000,5.000000,11.000000,23.000000,2018.000000,5.000000,11.000000,24.000000,2018.000000,2018-05-01 00:00:00
max,9800.000000,2018-12-30 00:00:00,2019-01-05 00:00:00,22638.480000,7.000000,12.000000,31.000000,2018.000000,6.000000,12.000000,31.000000,2019.000000,2018-12-01 00:00:00
std,2829.160653,NaN,NaN,626.651875,1.749614,3.281905,8.753733,1.123984,2.180441,3.337933,8.804986,1.126837,NaN


In [ ]:
df.isna().sum()

Only Postal Code has null data

In [ ]:
df[df['Postal Code'].isna()]

All empty rows have City - Burlington, so we can fill empty values with its Postal code

In [ ]:
# 05401 - Postal Code of Burlington, Vermont
df['Postal Code'] = df['Postal Code'].fillna('05401')

In [ ]:
df.isna().sum()

In [ ]:
df.nunique()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9,3))
df.groupby('Ship Mode')['Order ID'].nunique().sort_values(ascending = False).plot(kind='bar', ax=axes[0], color = '#CCCCFF', edgecolor = '#9966CC')
axes[0].set_title("Orders by Ship Mode")

df.groupby('Ship Mode')['Sales'].sum().sort_values(ascending = False).plot(kind='bar', ax=axes[1], color = '#CCCCFF', edgecolor = '#9966CC')
axes[1].set_title("Sales by Ship Mode")

df.groupby(['Order ID', 'Ship Mode'])['Sales'].sum().reset_index().groupby('Ship Mode')['Sales'].mean().sort_values(ascending = False).plot(kind='bar', ax=axes[2], color = '#CCCCFF', edgecolor = '#9966CC')
axes[2].set_title("Average price per order by Ship Mode")

Most orders (and profit) are delivering by Standart class, this can de explained by its low price

In [ ]:
plt.figure(figsize=(5,12))
df.groupby(['Order_month', 'Ship Mode'])['Order ID'].nunique().unstack().plot(kind='line',  color = ['#CCCCFF','#9966CC', '#CC99FF', '#9900CC'])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9,3))
df.groupby('Segment')['Order ID'].nunique().sort_values(ascending = False).plot(kind='pie', ax=axes[0], colors = ['#CCCCFF','#9966CC', '#CC99FF'], labeldistance = 0.3, autopct='%1.1f%%', textprops={'fontsize': 9})
axes[0].set_title("Orders by Segment")

df.groupby('Segment')['Sales'].sum().sort_values(ascending = False).plot(kind='bar', ax=axes[1], color = '#CCCCFF', edgecolor = '#9966CC')
axes[1].set_title("Sales by Segment")

df.groupby(['Order ID', 'Segment'])['Sales'].sum().reset_index().groupby('Segment')['Sales'].mean().sort_values(ascending = False).plot(kind='bar', ax=axes[2], color = '#CCCCFF', edgecolor = '#9966CC')
axes[2].set_title("Average price per order by Segment")

The biggest segment - Consumer, which also the cheapest (in avg per order price)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
df.groupby('Category')['Order ID'].nunique().sort_values(ascending = False).plot(kind='pie', ax=axes[0], colors = ['#CCCCFF','#9966CC', '#CC99FF'], labeldistance = 0.3, autopct='%1.1f%%', textprops={'fontsize': 9})
axes[0].set_title("Orders by Category")

df.groupby('Category')['Sales'].sum().sort_values(ascending=False).plot(kind='bar', ax=axes[1], color = '#CCCCFF', edgecolor = '#9966CC')
axes[1].set_title("Sales by Category")

df.groupby(['Order ID', 'Category'])['Sales'].sum().reset_index().groupby('Category')['Sales'].mean().sort_values(
    ascending=False).plot(kind='bar', ax=axes[2], color = '#CCCCFF', edgecolor = '#9966CC')
axes[2].set_title("Average price per order by Category")

Here, situation is similar to segments, the cheapest category is the most popular

In [ ]:
plt.figure(figsize=(5,12))
df.groupby(['Order_month', 'Category'])['Order ID'].nunique().unstack().plot(kind='line',  color = ['#CCCCFF','#9966CC', '#CC99FF', '#9900CC'])

In [ ]:
px.sunburst(df,
    path=['Category', 'Sub-Category'],
    values='Sales')

The biggest Sub-Category, as we can see, is Phones, which have 327782 sales

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
df.groupby('Region')['Order ID'].nunique().sort_values(ascending=False).plot(kind='bar', ax=axes[0], color = '#CCCCFF', edgecolor = '#9966CC')
axes[0].set_title("Orders by Region")

df.groupby('Region')['Sales'].sum().sort_values(ascending=False).plot(kind='bar', ax=axes[1], color = '#CCCCFF', edgecolor = '#9966CC')
axes[1].set_title("Sales by Region")

df.groupby(['Order ID', 'Region'])['Sales'].sum().reset_index().groupby('Region')['Sales'].mean().sort_values(
    ascending=False).plot(kind='bar', ax=axes[2], color = '#CCCCFF', edgecolor = '#9966CC')
axes[2].set_title("Region price per order by Category")

Most orders and sales were made from the West, despite the fact that the biggest price per order - from East

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
plt.subplots_adjust(hspace=1)
df.groupby('State')['Sales'].sum().sort_values(ascending=False).plot(kind='bar', ax=axes[0], color = '#CCCCFF', edgecolor = '#9966CC')
axes[0].set_title('Sales by States')

df.groupby('State')['Order ID'].nunique().sort_values(ascending=False).plot(kind='bar', ax=axes[1], color = '#CCCCFF', edgecolor = '#9966CC')
axes[1].set_title('Orders by States')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
plt.subplots_adjust(hspace=1)
df.groupby('City')['Sales'].sum().sort_values(ascending=False).head(10).plot(kind='bar', ax=axes[0], color = '#CCCCFF', edgecolor = '#9966CC')
axes[0].set_title('Top-10 Cities by Sales')

df.groupby('City')['Sales'].sum().sort_values(ascending=True).head(10).plot(kind='bar', ax=axes[1], color = '#CCCCFF', edgecolor = '#9966CC')
axes[1].set_title('Bottom-10 Cities by Sales')

In [ ]:
fig, axes =  plt.subplots(1, 2, figsize=(12, 3))

df.groupby('Order_weekday')['Order ID'].nunique().plot(kind='bar',  color = '#CCCCFF', edgecolor = '#9966CC', ax = axes[0])
axes[0].set_title('Orders by weekdays')

plt.figure(figsize=(3, 3))
df.groupby('Order_weekday')['shipping_time'].mean().plot(kind='bar',  color = '#CCCCFF', edgecolor = '#9966CC', ax = axes[1])
axes[1].set_title('Average shipping days by orders weekdays')

Here can be noticed a big drop in Thursday, but shipping time on Thursday is similar to over days

In [ ]:
fig, axes =  plt.subplots(1, 2, figsize=(12, 3))

df.groupby('Order_month')['Order ID'].nunique().plot(kind='bar',  color = '#CCCCFF', edgecolor = '#9966CC', ax = axes[0])
axes[0].set_title('Orders by months')

plt.figure(figsize=(3, 3))
df.groupby('Order_month')['shipping_time'].mean().plot(kind='bar',  color = '#CCCCFF', edgecolor = '#9966CC', ax = axes[1])
axes[1].set_title('Average shipping days by orders months')

The situation here is similar to line plots, here's a growth in the end of the year